# LeetCode 717 — 1-bit and 2-bit Characters bằng Python

**Link đề:** https://leetcode.com/problems/1-bit-and-2-bit-characters/
**Độ khó:** Easy · **Chủ đề:** Array

```python
class Solution:
    def isOneBitCharacter(self, bits: List[int]) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu bài toán **giải mã (decode)** một dãy bit theo bảng mã có độ dài thay đổi: `0` (1 bit), `10` / `11` (2 bit).
2. Nhận ra bảng mã này là **prefix code**: đọc từ trái sang phải, bit đầu quyết định độ dài ký tự → cách giải mã là **duy nhất**.
3. Cài 4 cách: thử mọi cách tách (đệ quy) → **con trỏ nhảy** `i += bits[i] + 1` → đếm **số bit 1 liên tiếp** trước bit 0 cuối (tính chẵn/lẻ) → DP.
4. Trace từng bước con trỏ, chứng minh đúng, Time/Space, các lỗi kinh điển (duyệt quá cuối, đếm sai đoạn 1).
5. Liên hệ: mã UTF-8, bài 91 Decode Ways, bài "jump" theo giá trị.

## 1. Đề bài và ví dụ

Có hai loại ký tự đặc biệt:
- Ký tự thứ nhất biểu diễn bằng **một bit** `0`.
- Ký tự thứ hai biểu diễn bằng **hai bit** `10` hoặc `11`.

Cho mảng nhị phân `bits` **kết thúc bằng `0`**. Trả `True` nếu ký tự cuối cùng **bắt buộc** là ký tự 1-bit.

| `bits` | Cách giải mã | Kết quả |
|---|---|:---:|
| `[1,0,0]` | `10` \| `0` | `True` |
| `[1,1,1,0]` | `11` \| `10` | `False` |

**Constraints:** `1 <= len(bits) <= 1000`; `bits[i]` là `0` hoặc `1`; `bits` kết thúc bằng `0` (theo đề).

In [1]:
for bits in [[1, 0, 0], [1, 1, 1, 0]]:
    print(bits, "| độ dài:", len(bits), "| bit cuối:", bits[-1])

[1, 0, 0] | độ dài: 3 | bit cuối: 0
[1, 1, 1, 0] | độ dài: 4 | bit cuối: 0


## 2. Hiểu sâu bảng mã: vì sao giải mã là duy nhất?

Nhìn **bit đầu** của một ký tự:
- Bit đầu là `0` → ký tự **phải** là `0` (1 bit), vì cả hai mã 2-bit đều bắt đầu bằng `1`.
- Bit đầu là `1` → ký tự **phải** là 2 bit (`10` hoặc `11`), vì không có mã 1-bit nào bắt đầu bằng `1`.

Không mã nào là **tiền tố (prefix)** của mã khác → gọi là **prefix code**. Nhờ đó đọc từ trái sang phải chỉ có **một** cách tách. Bài toán chỉ còn: tách xong, ký tự cuối có phải `0` đứng một mình không?

Ví dụ tay `[1, 1, 0, 1, 0, 0]`:
```text
vị trí 0: bit 1 -> ký tự 2-bit "11" (vị trí 0,1)
vị trí 2: bit 0 -> ký tự 1-bit "0"
vị trí 3: bit 1 -> ký tự 2-bit "10" (vị trí 3,4)
vị trí 5: bit 0 -> ký tự 1-bit "0"   <- ký tự cuối là 1-bit -> True
```

## 3. Nền tảng Python

### 3.1. Indexing và `len`
`bits[i]` đọc bit ở vị trí `i`; chỉ số cuối là `len(bits) - 1`; `bits[-1]` là bit cuối.

### 3.2. `while` với con trỏ tự nhảy
Khi bước nhảy **không cố định** (1 hoặc 2), dùng `while i < n:` và tự cập nhật `i`, thay vì `for i in range(n)` (luôn tăng 1).

In [2]:
bits = [1, 1, 0, 1, 0, 0]
i = 0
while i < len(bits):
    step = 2 if bits[i] == 1 else 1
    print(f"i={i}, bits[i]={bits[i]} -> ký tự {bits[i:i + step]} dài {step}")
    i += step

i=0, bits[i]=1 -> ký tự [1, 1] dài 2
i=2, bits[i]=0 -> ký tự [0] dài 1
i=3, bits[i]=1 -> ký tự [1, 0] dài 2
i=5, bits[i]=0 -> ký tự [0] dài 1


### 3.3. Mẹo `bits[i] + 1`
Vì `bits[i]` là `0` hoặc `1`: độ dài ký tự = `bits[i] + 1` (0 → 1 bit, 1 → 2 bit). Ta có thể viết `i += bits[i] + 1` không cần `if`.

In [3]:
for bit in [0, 1]:
    print("bit đầu", bit, "-> độ dài ký tự", bit + 1)

bit đầu 0 -> độ dài ký tự 1
bit đầu 1 -> độ dài ký tự 2


### 3.4. Slicing `bits[i:j]` và list rỗng là "falsy"
`bits[i:j]` lấy đoạn con; `not []` là `True`. Dùng trong cách đệ quy.

## 4. Cách 1 — Thử mọi cách tách bằng đệ quy (baseline)

Không dùng nhận xét "prefix code", người mới có thể nghĩ: "liệt kê **mọi** cách tách hợp lệ, xem ký tự cuối của chúng". Đề nói "bắt buộc là 1-bit" → mọi cách tách hợp lệ đều kết thúc bằng `0` đơn.

Hàm `parses(bits)` trả về list các cách tách (mỗi cách là list các ký tự dạng chuỗi).

In [4]:
def parses(bits):
    if not bits:
        return [[]]
    result = []
    if bits[0] == 0:
        for rest in parses(bits[1:]):
            result.append(["0"] + rest)
    if len(bits) >= 2 and bits[0] == 1:
        code = "1" + str(bits[1])
        for rest in parses(bits[2:]):
            result.append([code] + rest)
    return result

print(parses([1, 0, 0]))
print(parses([1, 1, 1, 0]))
print(parses([1, 1, 0, 1, 0, 0]))

[['10', '0']]
[['11', '10']]
[['11', '0', '10', '0']]


In [5]:
def one_bit_brute(bits):
    all_parses = parses(bits)
    return len(all_parses) > 0 and all(p[-1] == "0" for p in all_parses)

print(one_bit_brute([1, 0, 0]), one_bit_brute([1, 1, 1, 0]))

True False


Quan sát: luôn có **đúng một** cách tách — xác nhận nhận xét prefix code. Đệ quy này mỗi lần cắt `bits[1:]` tốn `O(n)` → tổng `O(n²)`, lãng phí vì không cần thử nhiều nhánh.

`all(...)`: trả `True` nếu **mọi** phần tử đúng.

## 5. Nút thắt và insight

Vì giải mã duy nhất, ta không cần liệt kê; chỉ cần **đi theo** cách tách duy nhất và xem ký tự cuối **bắt đầu ở đâu**:
- Nếu ký tự cuối bắt đầu ở vị trí `n - 1` → nó là `0` đơn → `True`.
- Nếu ký tự cuối bắt đầu ở `n - 2` → nó là `10` → `False`.

Đúng như hint của đề: *"Theo dõi nơi ký tự kế tiếp bắt đầu. Cuối cùng, xem có bắt đầu ở bit cuối không."*

Thậm chí chỉ cần dừng **trước** bit cuối: duyệt khi `i < n - 1`; nếu dừng đúng tại `i == n - 1` thì bit cuối là một ký tự riêng.

## 6. Cách 2 — Con trỏ nhảy (lời giải chính)

Mổ xẻ:
1. `i = 0`, `n = len(bits)`.
2. `while i < n - 1:` — còn ít nhất một bit **trước** bit cuối chưa xử lý.
3. `i += bits[i] + 1` — nhảy qua ký tự bắt đầu tại `i`.
4. Kết thúc: `i == n - 1` (bắt đầu đúng ở bit cuối → `True`) hoặc `i == n` (ký tự 2-bit đã "nuốt" bit cuối → `False`).

In [6]:
def one_bit_pointer(bits):
    i, n = 0, len(bits)
    while i < n - 1:
        i += bits[i] + 1
    return i == n - 1

for bits in [[1, 0, 0], [1, 1, 1, 0], [0], [1, 1, 0], [1, 0, 1, 0, 0], [1, 1, 0, 1, 0, 0]]:
    print(bits, "->", one_bit_pointer(bits))

[1, 0, 0] -> True
[1, 1, 1, 0] -> False
[0] -> True
[1, 1, 0] -> True
[1, 0, 1, 0, 0] -> True
[1, 1, 0, 1, 0, 0] -> True


Vì sao `i` không thể vượt quá `n`? Trong vòng lặp `i <= n - 2`, mỗi bước nhảy tối đa 2 → `i <= n`. Vậy sau vòng lặp `i` là `n - 1` hoặc `n`.

## 7. TRACE từng bước

**`bits = [1, 1, 1, 0]`** (`n = 4`, dừng khi `i >= 3`):

| Bước | i | bits[i] | Ký tự | i mới |
|---|:---:|:---:|---|:---:|
| 1 | 0 | 1 | `11` (vị trí 0–1) | 2 |
| 2 | 2 | 1 | `10` (vị trí 2–3) | 4 |
| dừng | 4 | | `i == n` ≠ `n-1` | **False** |

**`bits = [1, 0, 1, 0, 0]`** (`n = 5`):

| Bước | i | bits[i] | Ký tự | i mới |
|---|:---:|:---:|---|:---:|
| 1 | 0 | 1 | `10` | 2 |
| 2 | 2 | 1 | `10` | 4 |
| dừng | 4 | | `i == n - 1` | **True** |

In [7]:
def trace_one_bit(bits):
    i, n = 0, len(bits)
    step = 1
    print(f"bits={bits}, n={n}, vòng lặp chạy khi i < {n - 1}")
    while i < n - 1:
        length = bits[i] + 1
        print(f"  bước {step}: i={i}, bits[i]={bits[i]} -> ký tự {bits[i:i + length]} -> i = {i + length}")
        i += length
        step += 1
    res = i == n - 1
    print(f"  dừng với i={i}: {'bắt đầu đúng ở bit cuối' if res else 'bit cuối đã thuộc ký tự 2-bit'} -> {res}")
    return res

trace_one_bit([1, 1, 1, 0])
trace_one_bit([1, 0, 1, 0, 0])
trace_one_bit([0])

bits=[1, 1, 1, 0], n=4, vòng lặp chạy khi i < 3
  bước 1: i=0, bits[i]=1 -> ký tự [1, 1] -> i = 2
  bước 2: i=2, bits[i]=1 -> ký tự [1, 0] -> i = 4
  dừng với i=4: bit cuối đã thuộc ký tự 2-bit -> False
bits=[1, 0, 1, 0, 0], n=5, vòng lặp chạy khi i < 4
  bước 1: i=0, bits[i]=1 -> ký tự [1, 0] -> i = 2
  bước 2: i=2, bits[i]=1 -> ký tự [1, 0] -> i = 4
  dừng với i=4: bắt đầu đúng ở bit cuối -> True
bits=[0], n=1, vòng lặp chạy khi i < 0
  dừng với i=0: bắt đầu đúng ở bit cuối -> True


True

## 8. Code nộp LeetCode

In [8]:
from typing import List

class Solution:
    def isOneBitCharacter(self, bits: List[int]) -> bool:
        i, n = 0, len(bits)
        while i < n - 1:
            i += bits[i] + 1
        return i == n - 1

sol = Solution()
print(sol.isOneBitCharacter([1, 0, 0]))      # True
print(sol.isOneBitCharacter([1, 1, 1, 0]))   # False

True
False


## 9. Cách 3 — Đếm số bit 1 liên tiếp ngay trước bit 0 cuối (tham lam)

Insight mạnh hơn: chỉ phần **cuối** của mảng quyết định.

- Tìm bit `0` **gần cuối nhất** đứng **trước** bit cuối (ở vị trí `j < n - 1`), hoặc coi `j = -1` nếu không có. Một bit `0` luôn **kết thúc** một ký tự (nó là `0` đơn hoặc là bit thứ hai của `10`). Vậy ký tự kế tiếp bắt đầu ở `j + 1`.
- Từ `j + 1` đến `n - 2` toàn là bit `1`, có `k = n - 2 - j` bit. Các bit `1` này ghép **từng cặp** thành `11`.
  - `k` **chẵn** → ghép hết, bit cuối `0` đứng một mình → `True`.
  - `k` **lẻ** → còn một bit `1` ghép với bit cuối thành `10` → `False`.

In [9]:
def one_bit_parity(bits):
    n = len(bits)
    k = 0
    i = n - 2
    while i >= 0 and bits[i] == 1:
        k += 1
        i -= 1
    return k % 2 == 0

for bits in [[1, 0, 0], [1, 1, 1, 0], [0], [1, 1, 0], [0, 1, 1, 1, 1, 0], [1, 0, 1, 1, 1, 0]]:
    print(bits, "| số 1 liên tiếp trước bit cuối:", end=" ")
    k = 0
    for x in reversed(bits[:-1]):
        if x != 1:
            break
        k += 1
    print(k, "->", one_bit_parity(bits))

[1, 0, 0] | số 1 liên tiếp trước bit cuối: 0 -> True
[1, 1, 1, 0] | số 1 liên tiếp trước bit cuối: 3 -> False
[0] | số 1 liên tiếp trước bit cuối: 0 -> True
[1, 1, 0] | số 1 liên tiếp trước bit cuối: 2 -> True
[0, 1, 1, 1, 1, 0] | số 1 liên tiếp trước bit cuối: 4 -> True
[1, 0, 1, 1, 1, 0] | số 1 liên tiếp trước bit cuối: 3 -> False


`reversed(bits[:-1])`: duyệt ngược các bit **trừ** bit cuối. `bits[:-1]` là mọi phần tử trừ phần tử cuối.

Vì sao bit `0` ở `j` luôn kết thúc một ký tự? Nếu `0` là bit **đầu** của ký tự thì ký tự đó là `0` đơn (kết thúc ngay). Nếu `0` là bit **thứ hai** thì ký tự là `10` (cũng kết thúc tại đó). Bit `0` không bao giờ là bit đầu của ký tự 2-bit.

Cách này duyệt từ cuối, dừng sớm: thường nhanh hơn khi mảng dài mà đoạn 1 cuối ngắn.

## 10. Cách 4 — Quy hoạch động (DP) "vị trí nào là điểm bắt đầu ký tự"

Cách tổng quát (dùng lại được cho bảng mã khác): `start[i] = True` nếu có một ký tự bắt đầu ở `i` theo cách tách từ đầu.
- `start[0] = True`.
- Nếu `start[i]` thì `start[i + bits[i] + 1] = True`.

Đáp án: `start[n - 1]`.

In [10]:
def one_bit_dp(bits):
    n = len(bits)
    start = [False] * (n + 1)
    start[0] = True
    for i in range(n):
        if start[i]:
            nxt = i + bits[i] + 1
            if nxt <= n:
                start[nxt] = True
    return start[n - 1]

print(one_bit_dp([1, 0, 0]), one_bit_dp([1, 1, 1, 0]), one_bit_dp([1, 0, 1, 0, 0]))

True False True


## 11. Vì sao đúng?

**Tính duy nhất (cơ sở của mọi cách):** bit đầu của ký tự quyết định độ dài (0 → 1 bit, 1 → 2 bit), nên từ một vị trí bắt đầu, ký tự kế tiếp được xác định duy nhất. Quy nạp: toàn bộ cách tách là duy nhất.

**Invariant của cách con trỏ:** *`i` luôn là vị trí bắt đầu của một ký tự trong cách tách duy nhất.*
- Ban đầu `i = 0` ✔.
- Nếu `i` là điểm bắt đầu, ký tự đó dài `bits[i] + 1`, nên `i + bits[i] + 1` là điểm bắt đầu kế tiếp ✔.
- Vòng lặp dừng khi `i >= n - 1`. Nếu `i == n - 1` thì có một ký tự bắt đầu ở bit cuối; vì `bits[-1] = 0`, ký tự đó là `0` đơn → `True`. Nếu `i == n` thì ký tự trước đó (bắt đầu ở `n - 2`) đã bao gồm bit cuối → ký tự cuối là `10` → `False`.

**Cách đếm chẵn/lẻ:** như phân tích ở mục 9 — bit `0` tại `j` kết thúc một ký tự, đoạn `k` bit `1` sau nó bị ép ghép cặp `11`, nên tính chẵn lẻ của `k` quyết định bit cuối có bị "kéo" vào cặp hay không.

## 12. Độ phức tạp và so sánh

Gọi `n = len(bits)`.

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Đệ quy liệt kê | thử mọi cách tách | `O(n²)` (do slicing) | `O(n²)` | Chỉ để hiểu đề |
| 2. Con trỏ nhảy | `i += bits[i] + 1` | `O(n)` | `O(1)` | **Nên nộp**, đúng hint |
| 3. Đếm 1 trước bit cuối | chẵn/lẻ | `O(n)` xấu nhất, thường nhanh hơn | `O(1)` | Khéo, cần chứng minh |
| 4. DP điểm bắt đầu | mảng `start` | `O(n)` | `O(n)` | Tổng quát cho mã khác |

## 13. Bộ test — mẫu, biên, phản ví dụ, đối chiếu

In [11]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:32} PASS ({actual})")

sol = Solution()
check("mẫu 1", sol.isOneBitCharacter([1, 0, 0]), True)
check("mẫu 2", sol.isOneBitCharacter([1, 1, 1, 0]), False)
check("chỉ [0]", sol.isOneBitCharacter([0]), True)
check("[1,0] (một ký tự 2-bit)", sol.isOneBitCharacter([1, 0]), False)
check("[0,0]", sol.isOneBitCharacter([0, 0]), True)
check("[1,1,0] (11 | 0)", sol.isOneBitCharacter([1, 1, 0]), True)
check("4 bit 1 chẵn trước 0", sol.isOneBitCharacter([0, 1, 1, 1, 1, 0]), True)
check("3 bit 1 lẻ trước 0", sol.isOneBitCharacter([0, 1, 1, 1, 0]), False)
check("phản ví dụ chỉ nhìn bits[-2]", sol.isOneBitCharacter([1, 1, 0]), True)

mẫu 1                            PASS (True)
mẫu 2                            PASS (False)
chỉ [0]                          PASS (True)
[1,0] (một ký tự 2-bit)          PASS (False)
[0,0]                            PASS (True)
[1,1,0] (11 | 0)                 PASS (True)
4 bit 1 chẵn trước 0             PASS (True)
3 bit 1 lẻ trước 0               PASS (False)
phản ví dụ chỉ nhìn bits[-2]     PASS (True)


In [12]:
from itertools import product
import random

checked = 0
for L in range(1, 13):
    for tup in product([0, 1], repeat=L - 1):
        bits = list(tup) + [0]
        exp = one_bit_brute(bits)
        assert one_bit_pointer(bits) == exp
        assert one_bit_parity(bits) == exp
        assert one_bit_dp(bits) == exp
        assert sol.isOneBitCharacter(bits) == exp
        checked += 1
print("Đã đối chiếu", checked, "mảng (mọi mảng kết thúc bằng 0, độ dài 1..12): 4 cách khớp brute force")

rng = random.Random(717)
for _ in range(300):
    bits = [rng.randint(0, 1) for _ in range(rng.randint(0, 999))] + [0]
    assert one_bit_pointer(bits) == one_bit_parity(bits) == one_bit_dp(bits)
print("300 mảng ngẫu nhiên dài tới 1000: khớp")

Đã đối chiếu 4095 mảng (mọi mảng kết thúc bằng 0, độ dài 1..12): 4 cách khớp brute force
300 mảng ngẫu nhiên dài tới 1000: khớp


## 14. Lỗi người mới hay gặp

### Lỗi 1 — Chỉ nhìn bit áp chót `bits[-2]`

In [13]:
def wrong_look_second_last(bits):
    return len(bits) == 1 or bits[-2] == 0

print("sai:", wrong_look_second_last([1, 1, 0]), "| đúng:", sol.isOneBitCharacter([1, 1, 0]))

sai: False | đúng: True


Vì sao sai: `[1, 1, 0]` có `bits[-2] = 1` nhưng tách là `11 | 0` → `True`. Bit `1` áp chót có thể là **bit thứ hai** của `11`. Phải biết ký tự **bắt đầu ở đâu**, không chỉ nhìn bit cạnh nhau. Sửa: đếm cả đoạn 1 (chẵn/lẻ) hoặc dùng con trỏ.

### Lỗi 2 — Vòng lặp `while i < n` rồi kiểm tra sai

In [14]:
def wrong_loop_to_end(bits):
    i, n = 0, len(bits)
    last_len = 0
    while i < n:
        last_len = bits[i] + 1
        i += last_len
    return i == n - 1           # SAI: sau vòng lặp i luôn = n

print("sai:", wrong_loop_to_end([1, 0, 0]), "| đúng:", sol.isOneBitCharacter([1, 0, 0]))

sai: False | đúng: True


Vì sao sai: duyệt hết thì `i` luôn bằng `n` (do bit cuối là 0 nên không vượt). Sửa: hoặc dừng ở `n - 1` (`while i < n - 1`), hoặc nhớ độ dài ký tự cuối và trả `last_len == 1`.

In [15]:
def fixed_loop_to_end(bits):
    i, n, last_len = 0, len(bits), 0
    while i < n:
        last_len = bits[i] + 1
        i += last_len
    return last_len == 1

print(fixed_loop_to_end([1, 0, 0]), fixed_loop_to_end([1, 1, 1, 0]))

True False


### Lỗi 3 — Đếm **tổng** số bit 1 thay vì số bit 1 **liên tiếp** sát bit cuối

In [16]:
def wrong_total_ones(bits):
    return bits[:-1].count(1) % 2 == 0

print("sai:", wrong_total_ones([1, 0, 1, 0]), "| đúng:", sol.isOneBitCharacter([1, 0, 1, 0]))

sai: True | đúng: False


Vì sao sai: `[1, 0, 1, 0]` có 2 bit 1 (chẵn) nhưng tách `10 | 10` → `False`. Bit `0` ở giữa "reset" việc ghép cặp. Sửa: chỉ đếm đoạn 1 **liền trước** bit cuối.

### Lỗi 4 — Dùng `for i in range(n)` rồi `i += 2` bên trong
Trong Python, gán lại `i` trong thân `for` **không** ảnh hưởng vòng lặp kế tiếp (vì `range` tự cấp giá trị mới). Phải dùng `while`.

In [17]:
for i in range(4):
    print("i =", i, end="; ")
    i += 2                      # không có tác dụng với vòng kế tiếp
print()

i = 0; i = 1; i = 2; i = 3; 


## 15. Mẹo nhận dạng pattern và biến thể

- **Mã độ dài thay đổi mà phần đầu quyết định độ dài** (prefix code) → duyệt bằng con trỏ `i += độ_dài(i)`. Ví dụ ngoài đời: **UTF-8** (byte đầu cho biết ký tự dài 1–4 byte).
- **"Chỉ phần cuối quyết định"** → thử duyệt ngược và tìm "điểm neo" (ở đây là bit `0` gần cuối).
- **Chẵn/lẻ của một đoạn ghép cặp** → `k % 2`.

Bài liên quan: 91 Decode Ways (mã **không** prefix → nhiều cách tách → DP đếm), 55 Jump Game (nhảy theo giá trị), 393 UTF-8 Validation.

### 15.1. Biến thể A — giải mã ra danh sách ký tự

In [18]:
def decode(bits):
    res, i = [], 0
    while i < len(bits):
        if bits[i] == 0:
            res.append("A")            # ký tự 1-bit
            i += 1
        else:
            res.append("B" if bits[i + 1] == 0 else "C")   # 10 -> B, 11 -> C
            i += 2
    return "".join(res)

print(decode([1, 1, 0, 1, 0, 0]), decode([1, 0, 0]), decode([1, 1, 1, 0]))

CABA BA CB


### 15.2. Biến thể B — kiểm tra UTF-8 đơn giản (ý tưởng của bài 393)
Byte đầu `0xxxxxxx` → 1 byte; `110xxxxx` → 2; `1110xxxx` → 3; `11110xxx` → 4; các byte tiếp theo phải là `10xxxxxx`. Cùng khuôn "đọc đầu → biết độ dài → nhảy".

In [19]:
def char_len_utf8(byte):
    if byte >> 7 == 0b0: return 1
    if byte >> 5 == 0b110: return 2
    if byte >> 4 == 0b1110: return 3
    if byte >> 3 == 0b11110: return 4
    return -1

data = list("Hà".encode("utf-8"))
print(data, [format(x, "08b") for x in data])
i = 0
while i < len(data):
    L = char_len_utf8(data[i])
    print("ký tự bắt đầu tại", i, "dài", L, "byte")
    i += L

[72, 195, 160] ['01001000', '11000011', '10100000']
ký tự bắt đầu tại 0 dài 1 byte
ký tự bắt đầu tại 1 dài 2 byte


`"Hà".encode("utf-8")` đổi chuỗi thành dãy byte; `H` là 1 byte, `à` là 2 byte. `>>` là dịch phải bit (bỏ các bit thấp).

## 16. Bài tập tự luyện

**Bài 1 (tính tay).** Tách `[1, 1, 1, 1, 0, 1, 0, 0]` thành các ký tự và cho biết kết quả. Gán list ký tự (dạng chuỗi như `"11"`, `"0"`) vào `ex1_parse`, kết quả vào `ex1_answer`.

**Bài 2 (đoán output).** Với cách con trỏ, giá trị cuối của `i` khi chạy `[1, 0, 1, 0]` là bao nhiêu? Gán `ex2_i`.

**Bài 3 (TODO).** `ex3_is_one_bit(bits)`: tự viết cách con trỏ.

**Bài 4 (TODO).** `ex4_is_one_bit_parity(bits)`: tự viết cách đếm chẵn/lẻ.

**Bài 5 (biến thể).** `ex5_count_chars(bits)`: đếm số ký tự 1-bit và số ký tự 2-bit, trả tuple `(c1, c2)`.

**Bài 6 (mini challenge).** `ex6_count_parses(s)`: với bảng mã **không** prefix: `"0"` → A, `"1"` → B, `"10"` → C, `"11"` → D, đếm số cách tách chuỗi nhị phân `s` (giống Decode Ways). Ví dụ `"110"` có 3 cách: `1|1|0`, `11|0`, `1|10`.

In [20]:
ex1_parse = None    # TODO
ex1_answer = None   # TODO
ex2_i = None        # TODO

def ex3_is_one_bit(bits):
    # TODO
    pass

def ex4_is_one_bit_parity(bits):
    # TODO
    pass

# assert ex3_is_one_bit([1, 0, 0]) is True
# assert ex4_is_one_bit_parity([1, 1, 1, 0]) is False

In [21]:
def ex5_count_chars(bits):
    # TODO
    pass

def ex6_count_parses(s):
    # TODO
    pass

# assert ex5_count_chars([1, 1, 0, 1, 0, 0]) == (2, 2)
# assert ex6_count_parses("110") == 3

## 17. Gợi ý 3 mức

- **Mức 1:** Bit đầu quyết định độ dài; mọi bài đều là "đi từ trái, nhảy 1 hoặc 2".
- **Mức 2:** Bài 4: duyệt ngược từ `n - 2` đếm bit 1 tới khi gặp 0 hoặc hết mảng. Bài 6: `dp[i]` = số cách tách `s[:i]`; `dp[i] = dp[i-1]` (ký tự 1-bit cuối, luôn hợp lệ) `+ dp[i-2]` nếu `s[i-2] == '1'` (ký tự 2-bit cuối phải bắt đầu bằng 1).
- **Mức 3:** Bài 6:
  ```text
  dp = [1] + [0] * n
  for i in 1..n: dp[i] = dp[i-1] + (dp[i-2] if i >= 2 and s[i-2] == '1' else 0)
  ```

## 18. Đáp án, template nhớ nhanh và checklist

Bài 1: `11 | 11 | 0 | 10 | 0` → ký tự cuối là `0` → `True`.
Bài 2: `i: 0 → 2 → 4`; vòng dừng khi `i >= 3`, nên `i = 4` (= `n`) → `False`.

In [22]:
ans_ex1_parse = ["11", "11", "0", "10", "0"]
ans_ex1_answer = True
ans_ex2_i = 4

def ans_ex3_is_one_bit(bits):
    i = 0
    while i < len(bits) - 1:
        i += bits[i] + 1
    return i == len(bits) - 1

def ans_ex4_is_one_bit_parity(bits):
    k, i = 0, len(bits) - 2
    while i >= 0 and bits[i] == 1:
        k += 1
        i -= 1
    return k % 2 == 0

def ans_ex5_count_chars(bits):
    c1 = c2 = i = 0
    while i < len(bits):
        if bits[i] == 0:
            c1 += 1; i += 1
        else:
            c2 += 1; i += 2
    return c1, c2

def ans_ex6_count_parses(s):
    n = len(s)
    dp = [1] + [0] * n
    for i in range(1, n + 1):
        dp[i] = dp[i - 1]
        if i >= 2 and s[i - 2] == "1":
            dp[i] += dp[i - 2]
    return dp[n]

def brute_parses_ex6(s):
    if not s:
        return 1
    total = brute_parses_ex6(s[1:])
    if len(s) >= 2 and s[0] == "1":
        total += brute_parses_ex6(s[2:])
    return total

assert parses([1, 1, 1, 1, 0, 1, 0, 0]) == [ans_ex1_parse]
assert one_bit_pointer([1, 1, 1, 1, 0, 1, 0, 0]) == ans_ex1_answer
i = 0
while i < 3:
    i += [1, 0, 1, 0][i] + 1
assert i == ans_ex2_i
assert ans_ex5_count_chars([1, 1, 0, 1, 0, 0]) == (2, 2)
assert ans_ex6_count_parses("110") == 3
for L in range(1, 11):
    for tup in product([0, 1], repeat=L - 1):
        bits = list(tup) + [0]
        assert ans_ex3_is_one_bit(bits) == ans_ex4_is_one_bit_parity(bits) == one_bit_brute(bits)
        s = "".join(map(str, bits))
        assert ans_ex6_count_parses(s) == brute_parses_ex6(s)
print("Đáp án tham khảo: PASS toàn bộ")

Đáp án tham khảo: PASS toàn bộ


### Template nhớ nhanh

```python
class Solution:
    def isOneBitCharacter(self, bits: List[int]) -> bool:
        i, n = 0, len(bits)
        while i < n - 1:          # dừng TRƯỚC bit cuối
            i += bits[i] + 1      # 0 -> nhảy 1, 1 -> nhảy 2
        return i == n - 1         # bắt đầu đúng ở bit cuối?
```

Câu thần chú: **"Bit đầu quyết định độ dài; nhảy tới khi chạm bit cuối; đứng đúng ở bit cuối thì là 1-bit."**

### Checklist trước khi submit
- [ ] Dùng `while`, không dùng `for ... range` khi bước nhảy thay đổi.
- [ ] Điều kiện lặp `i < n - 1` và trả `i == n - 1`.
- [ ] Không chỉ nhìn `bits[-2]`; nếu dùng cách chẵn/lẻ thì chỉ đếm đoạn 1 liền trước bit cuối.
- [ ] Test `[0]`, `[1,0]`, `[1,1,0]`, `[1,0,1,0]`, đoạn 1 chẵn/lẻ.
- [ ] Time `O(n)`, Space `O(1)`.